# Music Annotation Project V8.5 — Notebook 44 REBUILT
## Read-only treatment-control delta audit against the exact Notebook 43 contract

**Runtime: CPU. Development diagnostic only.**

This notebook consumes one completed corrected Notebook 43 run and does not train, refit, recalibrate, alter thresholds, or modify source artefacts.

### Exact source experiment

- control: `control_33_97`
- treatment: `treatment_33_97_193`
- diagnostic question: where did the added 193×193 outer-context branch change safety/value confidence?

### Exact Notebook 43 file contract consumed here

Notebook 44 resolves a run below:

`MusicAnnotationProject/observed_print_classifier_v8_5_notebook43_rebuilt/<run>/`

and requires the exact root/table/audit artefacts generated by the rebuilt Notebook 43. It also requires `tables/02_candidate_group_pixels.parquet`, because exact candidate bounding boxes are derived from candidate pixels rather than guessed from centre coordinates.

For visual review, Notebook 44 reads the `observed.png` files written by Notebook 43 under both paired arms and verifies they are byte-identical before rendering a crop. It does not fall back to the mutable project-level `aligned/` folder.


In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!pip install -q pandas numpy pyarrow matplotlib opencv-python-headless


## 1. Configuration and exact Notebook 43 input contract


In [ ]:
from pathlib import Path
from datetime import datetime, timezone
import json, hashlib, math, shutil, sys, platform, zipfile
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt

ROOT=Path('/content/drive/MyDrive/MusicAnnotationProject')
N43_ROOT=ROOT/'observed_print_classifier_v8_5_notebook43_rebuilt'
OUTPUT_ROOT=ROOT/'observed_print_classifier_v8_5_notebook44_delta_audit'
SOURCE_N43_RUN_OVERRIDE=None
CONTROL_ARM='control_33_97'
TREATMENT_ARM='treatment_33_97_193'
EXPECTED_ARMS={CONTROL_ARM,TREATMENT_ARM}
TOP_N=60
CONTACT_N=24
CROP_MARGIN=150
EPS=1e-12

REQUIRED=[
 '01_pre_registration.json',
 'source_record.json',
 'candidate_lock.json',
 'SELF_VALIDATION_PASSED.json',
 'experiment_manifest.json',
 'notebook43_verdict.json',
 'executive_summary.txt',
 'tables/02_candidate_group_manifest.parquet',
 'tables/02_candidate_group_pixels.parquet',
 'tables/03_source_group_fold_map.csv',
 'tables/04_oof_safety_and_value_predictions.parquet',
 'tables/05_counterfactual_value_labels.parquet',
 'tables/arm_comparison_summary.csv',
 'tables/results_by_page.csv',
 'tables/results_by_source_group.csv',
 'audit/model_inventory.csv',
 'audit/calibrator_inventory.csv',
 'audit/page_output_integrity.csv',
]

def sha256_file(path):
 h=hashlib.sha256()
 with open(path,'rb') as f:
  for b in iter(lambda:f.read(1048576),b''):h.update(b)
 return h.hexdigest()

def complete_runs(root):
 hits=[]
 if root.exists():
  for p in root.iterdir():
   if p.is_dir() and all((p/r).exists() for r in REQUIRED):hits.append(p)
 return sorted(hits,key=lambda p:p.stat().st_mtime)

if SOURCE_N43_RUN_OVERRIDE:
 SOURCE=Path(SOURCE_N43_RUN_OVERRIDE)
else:
 hits=complete_runs(N43_ROOT)
 SOURCE=hits[-1] if hits else None
if SOURCE is None:raise RuntimeError(f'No complete corrected Notebook 43 run found under {N43_ROOT}')

RUN=datetime.now().strftime('%Y%m%d_%H%M%S')+'_notebook44_read_only_delta_audit'
OUT=OUTPUT_ROOT/RUN;TABLES=OUT/'tables';AUDIT=OUT/'audit';VISUAL=OUT/'visuals';REVIEW=OUT/'review';UPLOAD=OUT/'upload'
for p in [OUT,TABLES,AUDIT,VISUAL,REVIEW,UPLOAD]:p.mkdir(parents=True,exist_ok=False)
print('Notebook 43 source:',SOURCE)
print('Notebook 44 output:',OUT)


## 2. Preflight: print every required Notebook 43 input as FOUND/MISSING


In [ ]:
preflight=[]
for rel in REQUIRED:
 p=SOURCE/rel
 preflight.append({'relative_path':rel,'status':'FOUND' if p.exists() else 'MISSING','bytes':p.stat().st_size if p.exists() and p.is_file() else np.nan,'sha256':sha256_file(p) if p.exists() and p.is_file() else None})
preflight=pd.DataFrame(preflight)
preflight.to_csv(AUDIT/'notebook43_input_preflight.csv',index=False)
display(preflight)
missing=preflight.loc[preflight.status!='FOUND','relative_path'].tolist()
if missing:raise RuntimeError(f'Notebook 44 blocked: Notebook 43 inputs missing: {missing}')

pre=json.loads((SOURCE/'01_pre_registration.json').read_text())
selfv=json.loads((SOURCE/'SELF_VALIDATION_PASSED.json').read_text())
if pre.get('arms')!=[CONTROL_ARM,TREATMENT_ARM]:raise RuntimeError(f'Wrong Notebook 43 arms: {pre.get("arms")}')
if pre.get('single_experimental_variable')!='addition of the 193x193 isotropic outer-context branch':raise RuntimeError('Notebook 43 is not the corrected controlled experiment.')
if not all(v is True for v in selfv.values() if isinstance(v,bool)):raise RuntimeError(f'Notebook 43 self-validation did not fully pass: {selfv}')
print('PRE-FLIGHT PASSED')
print('CONTROL   :',CONTROL_ARM)
print('TREATMENT :',TREATMENT_ARM)


## 3. Load exact OOF schema, candidate manifest and exact candidate pixels


In [ ]:
oof=pd.read_parquet(SOURCE/'tables/04_oof_safety_and_value_predictions.parquet')
cand=pd.read_parquet(SOURCE/'tables/02_candidate_group_manifest.parquet')
pix=pd.read_parquet(SOURCE/'tables/02_candidate_group_pixels.parquet')
arm_summary=pd.read_csv(SOURCE/'tables/arm_comparison_summary.csv')

required_oof={
 'arm','row_id','group_id','page','source_group','candidate_group_id','mask_component_id',
 'raw_safety','raw_value','p_safety','p_value','pred_value',
 'safety_label','safety_label_eligible','value_positive','value_score',
 'truth_print_pixels','truth_annotation_pixels','truth_uncertain_pixels'
}
missing=required_oof-set(oof.columns)
if missing:raise RuntimeError(f'Notebook 43 OOF schema mismatch. Missing: {sorted(missing)}')
required_manifest={'group_id','page','candidate_group_id','centre_y','centre_x'}
missing=required_manifest-set(cand.columns)
if missing:raise RuntimeError(f'Notebook 43 candidate manifest missing: {sorted(missing)}')
required_pix={'group_id','page','y','x'}
missing=required_pix-set(pix.columns)
if missing:raise RuntimeError(f'Notebook 43 candidate pixel table missing: {sorted(missing)}')
if set(oof.arm.astype(str).unique())!=EXPECTED_ARMS:raise RuntimeError(f'Unexpected arms: {sorted(oof.arm.unique())}')
if oof.duplicated(['arm','row_id']).any():raise RuntimeError('Duplicate arm/row_id rows.')
if oof.duplicated(['arm','group_id']).any():raise RuntimeError('Duplicate arm/group_id rows.')

# Exact bounding boxes from candidate pixel coordinates.
bbox=(pix.groupby(['group_id','page'],as_index=False).agg(
 x0=('x','min'),x1=('x','max'),y0=('y','min'),y1=('y','max'),candidate_pixel_rows=('x','size')
))
bbox['bbox_w']=bbox.x1-bbox.x0+1;bbox['bbox_h']=bbox.y1-bbox.y0+1

manifest_geometry=cand[['group_id','page','candidate_group_id','centre_y','centre_x','pixels']].copy()
manifest_geometry=manifest_geometry.merge(bbox,on=['group_id','page'],how='left',validate='one_to_one')
if manifest_geometry[['x0','x1','y0','y1']].isna().any().any():raise RuntimeError('Some candidate groups lack exact candidate pixels.')
if 'pixels' in manifest_geometry and not np.array_equal(manifest_geometry.pixels.to_numpy(int),manifest_geometry.candidate_pixel_rows.to_numpy(int)):
 raise RuntimeError('Candidate manifest pixel count does not match candidate-pixel table.')
print('OOF rows:',len(oof),'candidate groups:',len(cand),'candidate pixels:',len(pix))


## 4. Prove exact control/treatment pairing by both row_id and group_id


In [ ]:
control=oof[oof.arm==CONTROL_ARM].sort_values('row_id').reset_index(drop=True)
treatment=oof[oof.arm==TREATMENT_ARM].sort_values('row_id').reset_index(drop=True)
if len(control)!=len(treatment):raise RuntimeError('Arm row counts differ.')
if not control.row_id.equals(treatment.row_id):raise RuntimeError('Control/treatment row_id identity differs.')
if not control.group_id.equals(treatment.group_id):raise RuntimeError('Control/treatment group_id identity differs.')

identity_cols=['row_id','group_id','page','source_group','candidate_group_id','mask_component_id','safety_label','safety_label_eligible','value_positive','value_score','truth_print_pixels','truth_annotation_pixels','truth_uncertain_pixels']
for c in identity_cols:
 if not control[c].equals(treatment[c]):raise RuntimeError(f'Control/treatment identity mismatch: {c}')

pair=control[identity_cols].copy()
for prefix,frame in [('control',control),('treatment',treatment)]:
 for c in ['raw_safety','raw_value','p_safety','p_value','pred_value','eligible_select','selected','reason']:
  if c in frame.columns:pair[prefix+'_'+c]=frame[c].to_numpy()
for c in ['raw_safety','raw_value','p_safety','p_value','pred_value']:
 pair['delta_'+c]=pair['treatment_'+c]-pair['control_'+c]
pair['abs_delta_p_value']=pair.delta_p_value.abs();pair['abs_delta_p_safety']=pair.delta_p_safety.abs();pair['move_l2']=np.hypot(pair.delta_p_value,pair.delta_p_safety)
pair['quadrant']=np.select([
 (pair.delta_p_value>EPS)&(pair.delta_p_safety>EPS),
 (pair.delta_p_value>EPS)&(pair.delta_p_safety<-EPS),
 (pair.delta_p_value<-EPS)&(pair.delta_p_safety>EPS),
 (pair.delta_p_value<-EPS)&(pair.delta_p_safety<-EPS),
],['VALUE_UP__SAFETY_UP','VALUE_UP__SAFETY_DOWN','VALUE_DOWN__SAFETY_UP','VALUE_DOWN__SAFETY_DOWN'],default='NO_MATERIAL_DIRECTION')
pair['safety_truth']=np.select([~pair.safety_label_eligible.astype(bool),pair.safety_label.astype(bool)],['TRUTH_UNCERTAIN','SAFE'],default='UNSAFE')
pair['value_truth']=np.where(pair.value_positive.astype(bool),'VALUE_POSITIVE','VALUE_NONPOSITIVE')
pair['truth_combo']=pair.safety_truth+'__'+pair.value_truth
pair=pair.merge(manifest_geometry,on=['group_id','page','candidate_group_id'],how='left',validate='one_to_one')
if pair[['x0','x1','y0','y1','centre_x','centre_y']].isna().any().any():raise RuntimeError('Pair geometry incomplete after manifest join.')
pair.to_parquet(TABLES/'paired_candidate_deltas.parquet',index=False);pair.to_csv(TABLES/'paired_candidate_deltas.csv',index=False)
print('Exact paired candidates:',len(pair))


## 5. Verify Notebook 43 observable whole-page provenance


In [ ]:
page_images={};image_audit=[]
for page in sorted(pair.page.unique()):
 cpath=SOURCE/'whole_pages'/CONTROL_ARM/page/'observed.png'
 tpath=SOURCE/'whole_pages'/TREATMENT_ARM/page/'observed.png'
 if not cpath.exists() or not tpath.exists():raise RuntimeError(f'{page}: per-arm Notebook 43 observed image missing')
 c=cv2.imread(str(cpath),0);t=cv2.imread(str(tpath),0)
 if c is None or t is None:raise RuntimeError(f'{page}: Notebook 43 observed image unreadable')
 if c.shape!=t.shape or not np.array_equal(c,t):raise RuntimeError(f'{page}: control/treatment observed page mismatch')
 page_images[page]=c
 image_audit.append({'page':page,'shape':str(c.shape),'control_observed_sha256':sha256_file(cpath),'treatment_observed_sha256':sha256_file(tpath),'byte_identical':sha256_file(cpath)==sha256_file(tpath)})
pd.DataFrame(image_audit).to_csv(AUDIT/'paired_observed_page_identity.csv',index=False)
print('Verified paired observed pages:',len(page_images))


## 6. Delta distributions and post-hoc truth/value strata


In [ ]:
def qstats(x):
 x=pd.Series(x).dropna().astype(float)
 q=x.quantile([0,.01,.05,.10,.25,.5,.75,.9,.95,.99,1])
 return {'n':len(x),'mean':x.mean(),'sd':x.std(ddof=1),'min':q.iloc[0],'p01':q.loc[.01],'p05':q.loc[.05],'p10':q.loc[.10],'p25':q.loc[.25],'median':q.loc[.5],'p75':q.loc[.75],'p90':q.loc[.9],'p95':q.loc[.95],'p99':q.loc[.99],'max':q.iloc[-1]}
rows=[]
for metric in ['delta_p_value','delta_p_safety','delta_pred_value','delta_raw_value','delta_raw_safety']:
 rows.append({'population':'ALL','metric':metric,**qstats(pair[metric])})
 for key,g in pair.groupby('truth_combo'):rows.append({'population':key,'metric':metric,**qstats(g[metric])})
pd.DataFrame(rows).to_csv(TABLES/'delta_distribution_summary.csv',index=False)
quad=pair.groupby(['quadrant','safety_truth','value_truth'],as_index=False).agg(groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),mean_value_score=('value_score','mean'),true_print_pixels=('truth_print_pixels','sum'),annotation_pixels=('truth_annotation_pixels','sum'))
quad.to_csv(TABLES/'quadrant_truth_summary.csv',index=False)
by_page=pair.groupby(['page','source_group'],as_index=False).agg(groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),value_up_safety_down=('quadrant',lambda x:int((x=='VALUE_UP__SAFETY_DOWN').sum())),both_up=('quadrant',lambda x:int((x=='VALUE_UP__SAFETY_UP').sum())),movement=('move_l2','sum'))
by_page.to_csv(TABLES/'delta_summary_by_page.csv',index=False)
by_source=pair.groupby('source_group',as_index=False).agg(groups=('row_id','size'),mean_delta_value=('delta_p_value','mean'),mean_delta_safety=('delta_p_safety','mean'),value_up_safety_down=('quadrant',lambda x:int((x=='VALUE_UP__SAFETY_DOWN').sum())),both_up=('quadrant',lambda x:int((x=='VALUE_UP__SAFETY_UP').sum())),movement=('move_l2','sum'))
by_source.to_csv(TABLES/'delta_summary_by_source_group.csv',index=False)
display(quad.sort_values('groups',ascending=False))


## 7. Paired ranking diagnostics


In [ ]:
def row(label,mask):
 q=pair[mask]
 return {'stratum':label,'groups':len(q),'mean_delta_p_value':q.delta_p_value.mean(),'median_delta_p_value':q.delta_p_value.median(),'fraction_value_up':float((q.delta_p_value>0).mean()),'mean_delta_p_safety':q.delta_p_safety.mean(),'median_delta_p_safety':q.delta_p_safety.median(),'fraction_safety_up':float((q.delta_p_safety>0).mean()),'fraction_value_up_safety_down':float((q.quadrant=='VALUE_UP__SAFETY_DOWN').mean())}
rank=pd.DataFrame([
 row('ALL',np.ones(len(pair),bool)),
 row('VALUE_POSITIVE',pair.value_positive.astype(bool)),
 row('VALUE_NONPOSITIVE',~pair.value_positive.astype(bool)),
 row('SAFE',pair.safety_truth=='SAFE'),
 row('UNSAFE',pair.safety_truth=='UNSAFE'),
 row('TRUTH_UNCERTAIN',pair.safety_truth=='TRUTH_UNCERTAIN'),
 row('SAFE_VALUE_POSITIVE',(pair.safety_truth=='SAFE')&pair.value_positive.astype(bool)),
 row('UNSAFE_VALUE_POSITIVE',(pair.safety_truth=='UNSAFE')&pair.value_positive.astype(bool)),
])
rank.to_csv(TABLES/'paired_ranking_diagnostics.csv',index=False);display(rank)


## 8. Predeclared review queues


In [ ]:
queues={
 '01_value_up_safety_down':pair[pair.quadrant=='VALUE_UP__SAFETY_DOWN'].sort_values(['delta_p_value','abs_delta_p_safety'],ascending=[False,False]),
 '02_both_up':pair[pair.quadrant=='VALUE_UP__SAFETY_UP'].sort_values('move_l2',ascending=False),
 '03_largest_safety_regressions':pair.sort_values('delta_p_safety'),
 '04_largest_safety_gains':pair.sort_values('delta_p_safety',ascending=False),
 '05_largest_value_gains':pair.sort_values('delta_p_value',ascending=False),
 '06_largest_value_regressions':pair.sort_values('delta_p_value'),
 '07_largest_joint_movements':pair.sort_values('move_l2',ascending=False),
 '08_safe_value_positive':pair[(pair.safety_truth=='SAFE')&pair.value_positive.astype(bool)].sort_values('move_l2',ascending=False),
 '09_unsafe_value_positive':pair[(pair.safety_truth=='UNSAFE')&pair.value_positive.astype(bool)].sort_values('move_l2',ascending=False),
}
for name,q in queues.items():q.head(TOP_N).to_csv(TABLES/f'{name}.csv',index=False)
pd.DataFrame([{'queue':k,'available':len(v),'exported':min(TOP_N,len(v))} for k,v in queues.items()]).to_csv(TABLES/'review_queue_summary.csv',index=False)


## 9. Observable contact sheets using exact Notebook 43 candidate extents


In [ ]:
def crop(row):
 im=page_images[row.page];H,W=im.shape
 x0=int(row.x0);x1=int(row.x1);y0=int(row.y0);y1=int(row.y1)
 cx=(x0+x1)//2;cy=(y0+y1)//2
 ax0=max(0,cx-CROP_MARGIN);ax1=min(W,cx+CROP_MARGIN+1);ay0=max(0,cy-CROP_MARGIN);ay1=min(H,cy+CROP_MARGIN+1)
 rgb=cv2.cvtColor(im[ay0:ay1,ax0:ax1],cv2.COLOR_GRAY2RGB)
 cv2.rectangle(rgb,(x0-ax0,y0-ay0),(x1-ax0,y1-ay0),(255,0,0),2)
 return rgb

def make_sheet(q,name,n=CONTACT_N,cols=4):
 q=q.head(n).reset_index(drop=True)
 if q.empty:return
 nr=math.ceil(len(q)/cols);fig,axes=plt.subplots(nr,cols,figsize=(5*cols,5*nr),squeeze=False)
 for ax in axes.flat:ax.axis('off')
 for i,r in q.iterrows():
  axes.flat[i].imshow(crop(r));axes.flat[i].set_title(f'{r.page} {r.group_id} | dV={r.delta_p_value:+.3f} dS={r.delta_p_safety:+.3f}\n{r.safety_truth} | {r.value_truth}',fontsize=8);axes.flat[i].axis('off')
 plt.tight_layout();plt.savefig(REVIEW/f'{name}.png',dpi=150,bbox_inches='tight');plt.close(fig)
for name,q in queues.items():make_sheet(q,name)
print('Contact sheets:',REVIEW)


## 10. Probability-delta plots


In [ ]:
plt.figure(figsize=(8,7))
for k,g in pair.groupby('safety_truth'):plt.scatter(g.delta_p_value,g.delta_p_safety,s=22,alpha=.65,label=k)
plt.axhline(0,color='black',lw=1);plt.axvline(0,color='black',lw=1);plt.xlabel('Treatment - control calibrated value probability');plt.ylabel('Treatment - control calibrated safety probability');plt.title('Notebook 44 paired deltas: added 193x193 context');plt.legend();plt.tight_layout();plt.savefig(VISUAL/'delta_scatter.png',dpi=180);plt.close()
fig,ax=plt.subplots(1,2,figsize=(14,5));ax[0].hist(pair.delta_p_value,bins=60);ax[0].axvline(0,color='black');ax[0].set_title('Delta p_value');ax[1].hist(pair.delta_p_safety,bins=60);ax[1].axvline(0,color='black');ax[1].set_title('Delta p_safety');plt.tight_layout();plt.savefig(VISUAL/'delta_histograms.png',dpi=180);plt.close()


## 11. Read-only diagnostic summary and integrity record


In [ ]:
a=arm_summary.set_index('arm');c=a.loc[CONTROL_ARM];t=a.loc[TREATMENT_ARM]
value_ap_delta=float(t.value_average_precision-c.value_average_precision);safety_ap_delta=float(t.safety_average_precision-c.safety_average_precision)
pattern='VALUE_RANKING_UP__SAFETY_RANKING_DOWN' if value_ap_delta>0 and safety_ap_delta<0 else ('BOTH_RANKING_METRICS_UP_OR_FLAT' if value_ap_delta>0 and safety_ap_delta>=0 else ('SAFETY_RANKING_UP__VALUE_RANKING_DOWN_OR_FLAT' if value_ap_delta<=0 and safety_ap_delta>0 else 'NO_RANKING_IMPROVEMENT'))
summary={'notebook':'44','read_only':True,'source_notebook43_run':str(SOURCE),'paired_groups':len(pair),'control_arm':CONTROL_ARM,'treatment_arm':TREATMENT_ARM,'control_value_average_precision':float(c.value_average_precision),'treatment_value_average_precision':float(t.value_average_precision),'delta_value_average_precision':value_ap_delta,'control_safety_average_precision':float(c.safety_average_precision),'treatment_safety_average_precision':float(t.safety_average_precision),'delta_safety_average_precision':safety_ap_delta,'control_selected_groups':int(c.selected_groups),'treatment_selected_groups':int(t.selected_groups),'observed_ranking_pattern':pattern,'quadrant_counts':{k:int(v) for k,v in pair.quadrant.value_counts().items()},'model_fit':False,'calibrator_fit':False,'threshold_search':False,'source_files_modified':False}
(OUT/'diagnostic_summary.json').write_text(json.dumps(summary,indent=2))
integrity={'exact_notebook43_contract':True,'preflight_passed':True,'row_id_pairing_exact':True,'group_id_pairing_exact':True,'candidate_bbox_from_exact_candidate_pixels':True,'observed_pages_from_notebook43_run':True,'paired_observed_pages_identical':True,'source_oof_sha256':sha256_file(SOURCE/'tables/04_oof_safety_and_value_predictions.parquet'),'source_candidate_pixels_sha256':sha256_file(SOURCE/'tables/02_candidate_group_pixels.parquet'),'no_retraining':True,'no_recalibration':True,'no_threshold_search':True,'source_files_modified':False}
(OUT/'READ_ONLY_INTEGRITY.json').write_text(json.dumps(integrity,indent=2))
lines=['NOTEBOOK 44 — READ-ONLY DELTA AUDIT AGAINST EXACT NOTEBOOK 43 CONTRACT','',f'Source: {SOURCE}',f'Paired candidate groups: {len(pair)}',f'Control: {CONTROL_ARM}',f'Treatment: {TREATMENT_ARM}','',f'Value AP delta: {value_ap_delta:+.6f}',f'Safety AP delta: {safety_ap_delta:+.6f}',f'Control selected groups: {int(c.selected_groups)}',f'Treatment selected groups: {int(t.selected_groups)}',f'Pattern: {pattern}','','No model fit, recalibration, threshold search or source modification was performed.']
(OUT/'executive_summary.txt').write_text(chr(10).join(lines));print(chr(10).join(lines))


## 12. Final upload package


In [ ]:
files=[OUT/'executive_summary.txt',OUT/'diagnostic_summary.json',OUT/'READ_ONLY_INTEGRITY.json',AUDIT/'notebook43_input_preflight.csv',AUDIT/'paired_observed_page_identity.csv',TABLES/'paired_candidate_deltas.csv',TABLES/'delta_distribution_summary.csv',TABLES/'quadrant_truth_summary.csv',TABLES/'paired_ranking_diagnostics.csv',TABLES/'delta_summary_by_page.csv',TABLES/'delta_summary_by_source_group.csv',TABLES/'review_queue_summary.csv',VISUAL/'delta_scatter.png',VISUAL/'delta_histograms.png']+sorted(REVIEW.glob('*.png'))
missing=[str(p) for p in files if not p.exists()]
if missing:raise RuntimeError(f'Upload artefacts missing: {missing}')
inv=[]
for p in files:
 rel=p.relative_to(OUT);dst=UPLOAD/rel;dst.parent.mkdir(parents=True,exist_ok=True);shutil.copy2(p,dst);inv.append({'relative_path':str(rel),'bytes':p.stat().st_size,'sha256':sha256_file(p)})
pd.DataFrame(inv).to_csv(UPLOAD/'upload_manifest.csv',index=False)
(UPLOAD/'README.txt').write_text('Notebook 44 rebuilt against the exact corrected Notebook 43 contract. Start with executive_summary.txt, READ_ONLY_INTEGRITY.json, audit/notebook43_input_preflight.csv, paired_ranking_diagnostics.csv, quadrant_truth_summary.csv, and review/01_value_up_safety_down.png.')
(UPLOAD/'upload_file_list.txt').write_text(chr(10).join(x['relative_path'] for x in inv))
zip_path=UPLOAD/'notebook44_EXACT_N43_CONTRACT_delta_audit.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
 for p in UPLOAD.rglob('*'):
  if p.is_file() and p!=zip_path:z.write(p,p.relative_to(UPLOAD))
print('Notebook 44 complete:',OUT);print('Upload bundle:',zip_path)
